# UTS Data Acquisition & Management
**Topik:** Kedatangan wisatawan mancanegara ke Bali per kebangsaan per bulan (2025) dan cuaca bulanan Bali
**Dosen:** Dr. I Gede Totok Suryawan, S.Kom., M.T. | **Kelas:** E | **Tanggal:** 19 November 2026

Tiga cara akuisisi: (a) berkas dataset terbuka, (b) API (Open-Meteo), (c) web scraping halaman statis (Wikipedia, daftar kecamatan di Bali).
Semua data mentah disimpan di `data/raw` tanpa diubah, dan setiap sumber dicatat di `docs/provenance.csv`.

## 0. Setup, konfigurasi, dan fungsi bantu
**Yang wajib kamu isi:** `NAMA`, `EMAIL_KAMPUS`.

In [11]:
import os, io, time, json, hashlib, datetime as dt
import requests
import pandas as pd
from urllib import robotparser
from urllib.parse import urlparse
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ====== ISI DATA DIRI ======
NAMA = "Ach. Rafli alfian syah"
NIM = "2501010048"
EMAIL_KAMPUS = "humas@instiki.ac.id."

# Pengaturan etika request (wajib: timeout, User-Agent berisi email, jeda)
TIMEOUT = 20      # detik
JEDA = 2          # detik jeda antar-request
USER_AGENT = f"UTS-Akuisisi-Data-INSTIKI/1.0 ({NAMA}; {EMAIL_KAMPUS})"

# Membuat folder kerja
for d in ["data/raw", "data/interim", "data/processed", "docs"]:
    os.makedirs(d, exist_ok=True)

# Session dengan retry otomatis untuk status code sementara (429/5xx)
def buat_session():
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})
    retry = Retry(total=3, backoff_factor=1,
                  status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["GET"])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://", HTTPAdapter(max_retries=retry))
    return s

session = buat_session()

def get_aman(url, params=None):
    # GET dengan jeda, timeout, dan pengecekan status code
    time.sleep(JEDA)
    r = session.get(url, params=params, timeout=TIMEOUT)
    print("Status:", r.status_code, "|", r.url)
    r.raise_for_status()
    return r

# Hash MD5 sebuah berkas (untuk bukti integritas)
def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(8192), b""):
            h.update(blok)
    return h.hexdigest()

# Catatan provenance: URL, tanggal akses, lisensi, MD5
PROV_PATH = "docs/provenance.csv"
def catat_provenance(sumber, path, url, lisensi, keterangan=""):
    baris = {
        "sumber": sumber, "file": path, "url": url,
        "tanggal_akses": dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "lisensi": lisensi, "md5": md5_file(path), "keterangan": keterangan,
    }
    if os.path.exists(PROV_PATH):
        prov = pd.read_csv(PROV_PATH)
        prov = prov[prov["file"] != path]          # hindari baris ganda saat sel dijalankan ulang
        prov = pd.concat([prov, pd.DataFrame([baris])], ignore_index=True)
    else:
        prov = pd.DataFrame([baris])
    prov.to_csv(PROV_PATH, index=False)
    print("Provenance tercatat:", path, "| MD5:", baris["md5"])

## Soal 1. Dataset berkas: baca, identifikasi, estimasi storage
Sumber: **Buku Statistik Pariwisata Bali 2025**, Dinas Pariwisata Provinsi Bali (file `Table-1-7-Final-5.xls`, sheet `tab7ok`:
kedatangan wisman langsung ke Bali per kebangsaan per bulan). Upload file ke `data/raw/` di Colab.
File `.xls` lama butuh paket `xlrd` (sel pertama di bawah).

In [12]:
!pip install -q xlrd

In [15]:
FILE_DATASET = "data/raw/Table-1-7-Final-5.xls"
SHEET = "tab7ok"
URL_DATASET = "https://disparda.baliprov.go.id/category/data-statistik/"
LISENSI_DATASET = ("Tidak ada lisensi eksplisit di berkas/portal; data publik Dinas Pariwisata Prov. Bali "
                   "(sumber data: BPS Prov. Bali), dipakai untuk keperluan akademik dengan mencantumkan sumber")

# Catat provenance SEBELUM berkas disentuh (hash = bukti data mentah tidak diubah)
catat_provenance("Dataset berkas (Statistik Pariwisata Bali 2025)", FILE_DATASET, URL_DATASET,
                 LISENSI_DATASET, "Sheet tab7ok; unduhan manual; ada catatan *TEMPORARY DATA* di berkas")

# Parameter baca: header=None karena judul/header sheet berlapis; na_values untuk tanda kosong; dtype=object agar tidak salah tebak
raw = pd.read_excel(FILE_DATASET, sheet_name=SHEET, header=None, skiprows=0,
                    na_values=["-", "NA", "n/a"], dtype=object)
print("Bentuk mentah sheet:", raw.shape)
raw.iloc[:12, :16]

Provenance tercatat: data/raw/Table-1-7-Final-5.xls | MD5: 7ff6f254fe4d4c2916f6b1b5cd3c2c43
Bentuk mentah sheet: (203, 19)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,DIRECT FOREIGN TOURIST ARRIVALS TO...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NO,NATIONALITY,JAN,FEB,MAR,APR,MAY,JUNE,JULY,AUG,SEP,OCT,NOV,DEC,TOTAL,%
3,I,ASIA PACIFIC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,EXCLUDING ASEAN,307053,242924,242180,308532,329882,349066,368385,350533,339880,324794,269820,316415,3749464,53.958796
5,1,Australian,134483,96421,103892,139481,138515,143915,161051,145756,152440,147581,123937,140987,1628459,23.435266
6,2,Bhutan,0,0,0,0,0,82,66,0,0,0,124,71,343,0.004936
7,3,Bangladesh,889,801,792,883,466,773,555,525,690,705,795,917,8791,0.126512
8,4,Fiji,0,0,86,0,167,99,77,0,122,142,132,139,964,0.013873
9,5,Hongkong,3181,2298,2763,5593,5582,7182,5914,5562,5287,5337,3384,4460,56543,0.813714


In [16]:
# Membersihkan sheet menjadi tabel rapi: satu baris per negara (kolom Jan..Des + total)
ROMAWI = {"I", "II", "III", "IV", "V", "VI", "VII"}
BULAN_KOL = ["jan","feb","mar","apr","mei","jun","jul","agu","sep","okt","nov","des"]

baris, benua = [], None
for _, r in raw.iloc[3:].iterrows():
    no = str(r[0]).strip()
    nama = r[1]
    if pd.isna(nama):
        continue
    nama = str(nama).strip()
    if nama.upper() == "TOTAL":                       # akhir tabel
        break
    if no in ROMAWI and no != "VII":                  # baris judul kawasan (subtotal), simpan nama kawasan
        benua = "ASIA PACIFIC EXCL. ASEAN" if no == "I" else nama.upper()
        continue
    if nama.upper().startswith("EXCLUDING"):          # subtotal lanjutan Asia Pasifik
        continue
    if no == "VII":
        benua = "OTHER"
    total = pd.to_numeric(r[14], errors="coerce")
    if pd.isna(total):
        continue
    nilai = [pd.to_numeric(r[i], errors="coerce") for i in range(2, 14)]
    baris.append([benua, nama] + nilai + [total])

df_file = pd.DataFrame(baris, columns=["kawasan", "negara"] + BULAN_KOL + ["total"])
df_file[BULAN_KOL + ["total"]] = df_file[BULAN_KOL + ["total"]].fillna(0).astype(int)

# Validasi: jumlah 12 bulan harus sama dengan total
selisih = (df_file[BULAN_KOL].sum(axis=1) - df_file["total"]).abs()
print("Baris x kolom:", df_file.shape, "| baris total tidak cocok:", int((selisih > 0).sum()))
assert len(df_file) >= 100
df_file.to_csv("data/interim/wisman_bali_2025_wide.csv", index=False)
df_file.head()

Baris x kolom: (185, 15) | baris total tidak cocok: 0


,kawasan,negara,jan,feb,mar,apr,mei,jun,jul,agu,sep,okt,nov,des,total
0,ASIA PACIFIC EXCL. ASEAN,Australian,134483,96421,103892,139481,138515,143915,161051,145756,152440,147581,123937,140987,1628459
1,ASIA PACIFIC EXCL. ASEAN,Bhutan,0,0,0,0,0,82,66,0,0,0,124,71,343
2,ASIA PACIFIC EXCL. ASEAN,Bangladesh,889,801,792,883,466,773,555,525,690,705,795,917,8791
3,ASIA PACIFIC EXCL. ASEAN,Fiji,0,0,86,0,167,99,77,0,122,142,132,139,964
4,ASIA PACIFIC EXCL. ASEAN,Hongkong,3181,2298,2763,5593,5582,7182,5914,5562,5287,5337,3384,4460,56543


In [17]:
# Profil dataset: tipe data, nilai hilang, ringkasan statistik
print(df_file.dtypes, "\n")
print("Nilai hilang per kolom:\n", df_file.isna().sum(), "\n")
display(df_file.describe().T)

kawasan    object
negara     object
jan         int64
feb         int64
mar         int64
apr         int64
mei         int64
jun         int64
jul         int64
agu         int64
sep         int64
okt         int64
nov         int64
des         int64
total       int64
dtype: object 

Nilai hilang per kolom:
 kawasan    0
negara     0
jan        0
feb        0
mar        0
apr        0
mei        0
jun        0
jul        0
agu        0
sep        0
okt        0
nov        0
des        0
total      0
dtype: int64 



,count,mean,std,min,25%,50%,75%,max
jan,185.0,2864.308108,11780.000941,0.0,0.0,0.0,1223.0,134483.0
feb,185.0,2436.200000,8909.537307,0.0,0.0,0.0,1029.0,96421.0
mar,185.0,2545.140541,9395.084358,0.0,0.0,0.0,1023.0,103892.0
apr,185.0,3195.789189,12193.779103,0.0,0.0,120.0,1155.0,139481.0
mei,185.0,3255.205405,12637.799485,0.0,0.0,117.0,1064.0,138515.0
jun,185.0,3447.935135,13030.879717,0.0,0.0,121.0,1186.0,143915.0
jul,185.0,3768.145946,14171.981577,0.0,0.0,166.0,1295.0,161051.0
agu,185.0,3691.167568,13304.491342,0.0,0.0,158.0,1109.0,145756.0
sep,185.0,3433.237838,13127.060105,0.0,0.0,117.0,1207.0,152440.0
okt,185.0,3215.421622,12548.567612,0.0,0.0,120.0,1205.0,147581.0


In [18]:
# ====== IDENTIFIKASI DATASET ======
JENIS_DATA = "Terstruktur (tabel baris-kolom dari Excel .xls, dibersihkan jadi tabel rapi)"
SATUAN = "Jumlah kunjungan (kedatangan wisman langsung ke Bali), per kebangsaan per bulan"
CAKUPAN_WILAYAH = "Provinsi Bali (pintu masuk langsung; kebangsaan/negara asal)"
CAKUPAN_WAKTU = "Januari-Desember 2025 (data sementara menurut catatan berkas)"

ukuran_mb = os.path.getsize(FILE_DATASET) / 1024**2

# Estimasi storage 1 tahun = ukuran berkas x 12 bulan, ditambah cadangan 30%
tahun_mb = ukuran_mb * 12
total_mb = tahun_mb * 1.30
ringkasan = pd.DataFrame({
    "Item": ["Jenis data", "Jumlah baris (negara)", "Jumlah kolom", "Satuan", "Cakupan wilayah",
             "Cakupan waktu", "Ukuran berkas (MB)", "Storage 1 tahun x12 (MB)", "Total + cadangan 30% (MB)"],
    "Nilai": [JENIS_DATA, df_file.shape[0], df_file.shape[1], SATUAN, CAKUPAN_WILAYAH,
              CAKUPAN_WAKTU, round(ukuran_mb, 4), round(tahun_mb, 4), round(total_mb, 4)],
})
display(ringkasan)
ringkasan.to_csv("docs/identifikasi_dataset.csv", index=False)

,Item,Nilai
0,Jenis data,Terstruktur (tabel baris-kolom dari Excel .xls...
1,Jumlah baris (negara),185
2,Jumlah kolom,15
3,Satuan,Jumlah kunjungan (kedatangan wisman langsung k...
4,Cakupan wilayah,Provinsi Bali (pintu masuk langsung; kebangsaa...
5,Cakupan waktu,Januari-Desember 2025 (data sementara menurut ...
6,Ukuran berkas (MB),0.1689
7,Storage 1 tahun x12 (MB),2.0273
8,Total + cadangan 30% (MB),2.6355


**Asumsi estimasi storage (tulis ulang dengan kata-katamu):**
1. Berkas diperbarui/diunduh ulang sekali per bulan dan tiap unduhan disimpan sebagai berkas baru (tidak menimpa).
2. Ukuran berkas tiap bulan dianggap sama dengan ukuran berkas saat ini (satu workbook berisi 7 sheet).
3. Cadangan 30% untuk pertumbuhan data, berkas hasil olahan (interim/processed), dan log.

## Soal 2. Lisensi, PII/quasi-identifier, dan Data Ethics Checklist
**Langkah manual:** buka halaman sumber (`URL_DATASET`) dan cari keterangan hak cipta/lisensi. Tulis temuanmu apa adanya.
Catatan: sheet `tab1ok` di berkas yang sama memuat nama direktur, alamat, dan telepon asosiasi (PII). Sheet itu **tidak dipakai**.

In [19]:
# Deteksi kolom PII / quasi-identifier dari nama kolom tabel yang dipakai
KATA_PII = ["nama", "alamat", "telp", "telepon", "email", "nik", "ktp", "lahir", "kontak", "direktur"]
KATA_QUASI = ["negara", "kawasan", "bulan", "tahun", "kecamatan", "desa", "usia", "gender"]

kolom = [str(c).lower() for c in df_file.columns]
pii = [c for c in kolom if any(k in c for k in KATA_PII)]
quasi = [c for c in kolom if any(k in c for k in KATA_QUASI)]
print("Kolom PII langsung  :", pii if pii else "tidak ada")
print("Quasi-identifier    :", quasi)
print("Nilai terkecil per sel (risiko sel kecil):", int(df_file[BULAN_KOL].min().min()))

Kolom PII langsung  : tidak ada
Quasi-identifier    : ['kawasan', 'negara']
Nilai terkecil per sel (risiko sel kecil): 0


In [20]:
# Data Ethics Checklist 9 butir. Periksa dan ubah Status/Bukti sesuai kondisi sebenarnya.
checklist = pd.DataFrame([
    ["1. Sumber data jelas dan sah", "Ya", "Dinas Pariwisata Prov. Bali (disparda.baliprov.go.id), Buku Statistik Pariwisata 2025"],
    ["2. Lisensi diperiksa dan penggunaan diizinkan", "Sebagian", "Tidak ditemukan lisensi eksplisit; data publik pemerintah, dipakai akademik + sitasi sumber"],
    ["3. Tujuan penggunaan sesuai (akademik)", "Ya", "UTS mata kuliah Data Acquisition & Management"],
    ["4. Tidak memuat PII langsung", "Ya", "Tabel yang dipakai hanya agregat per negara per bulan; sheet tab1ok (PII) tidak dipakai"],
    ["5. Quasi-identifier dinilai dan dimitigasi", "Ya", "Hanya negara + bulan (agregat); tidak ada data individu. Sel kecil dicatat sebagai risiko rendah"],
    ["6. Minimisasi data (hanya yang diperlukan)", "Ya", "Hanya sheet tab7ok yang diambil untuk analisis"],
    ["7. Kepatuhan robots.txt dan ketentuan situs", "Ya", "Lihat bukti robots.txt di Soal 4"],
    ["8. Request santun (timeout, UA berisi email, jeda)", "Ya", f"TIMEOUT={TIMEOUT}s, JEDA={JEDA}s, User-Agent: {USER_AGENT}"],
    ["9. Kredensial tidak ada di notebook; provenance dicatat", "Ya", "Open-Meteo tanpa API key; docs/provenance.csv"],
], columns=["Butir", "Status", "Bukti"])
checklist.to_csv("docs/data_ethics_checklist.csv", index=False)
display(checklist)

,Butir,Status,Bukti
0,1. Sumber data jelas dan sah,Ya,Dinas Pariwisata Prov. Bali (disparda.baliprov...
1,2. Lisensi diperiksa dan penggunaan diizinkan,Sebagian,Tidak ditemukan lisensi eksplisit; data publik...
2,3. Tujuan penggunaan sesuai (akademik),Ya,UTS mata kuliah Data Acquisition & Management
3,4. Tidak memuat PII langsung,Ya,Tabel yang dipakai hanya agregat per negara pe...
4,5. Quasi-identifier dinilai dan dimitigasi,Ya,Hanya negara + bulan (agregat); tidak ada data...
5,6. Minimisasi data (hanya yang diperlukan),Ya,Hanya sheet tab7ok yang diambil untuk analisis
6,7. Kepatuhan robots.txt dan ketentuan situs,Ya,Lihat bukti robots.txt di Soal 4
7,"8. Request santun (timeout, UA berisi email, j...",Ya,"TIMEOUT=20s, JEDA=2s, User-Agent: UTS-Akuisisi..."
8,9. Kredensial tidak ada di notebook; provenanc...,Ya,Open-Meteo tanpa API key; docs/provenance.csv


## Soal 3. Akuisisi melalui API (Open-Meteo, cuaca harian Bali 2025)
Open-Meteo gratis dan **tidak memerlukan API key**, jadi tidak ada kredensial di notebook.
Lokasi: sekitar Bandara Ngurah Rai (pintu masuk utama wisman). Periode: 1 Januari sampai 31 Desember 2025 (365 hari).

In [21]:
URL_API = "https://archive-api.open-meteo.com/v1/archive"
PARAMS = {
    "latitude": -8.7467, "longitude": 115.1668,    # perkiraan titik Bandara Ngurah Rai
    "start_date": "2025-01-01", "end_date": "2025-12-31",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Asia/Makassar",                    # WITA, zona waktu Bali
}

try:
    resp = get_aman(URL_API, PARAMS)                # retry 429/5xx sudah diatur di session
    path_api = "data/raw/openmeteo_bali_2025.json"
    with open(path_api, "wb") as f:
        f.write(resp.content)                       # simpan respons mentah apa adanya
    catat_provenance("API Open-Meteo (archive)", path_api, resp.url,
                     "CC BY 4.0 (Open-Meteo, penggunaan non-komersial/akademik)", "Cuaca harian 2025")
except requests.exceptions.HTTPError as e:
    print("HTTP error:", e)
except requests.exceptions.RequestException as e:
    print("Gagal koneksi/timeout:", e)

Status: 200 | https://archive-api.open-meteo.com/v1/archive?latitude=-8.7467&longitude=115.1668&start_date=2025-01-01&end_date=2025-12-31&daily=temperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Asia%2FMakassar
Provenance tercatat: data/raw/openmeteo_bali_2025.json | MD5: 0597cf5846cdefbe0039f1b542b184ba


In [22]:
# Normalisasi JSON -> tabel (satu baris per hari)
with open("data/raw/openmeteo_bali_2025.json", encoding="utf-8") as f:
    js = json.load(f)

df_api = pd.DataFrame(js["daily"]).rename(columns={
    "time": "tanggal", "temperature_2m_max": "suhu_max_c",
    "temperature_2m_min": "suhu_min_c", "precipitation_sum": "hujan_mm"})
df_api["tanggal"] = pd.to_datetime(df_api["tanggal"])
assert len(df_api) >= 30, "Tabel API harus minimal 30 baris"
print("Baris x kolom:", df_api.shape)
df_api.to_csv("data/interim/cuaca_bali_2025_harian.csv", index=False)
df_api.head()

Baris x kolom: (365, 4)


,tanggal,suhu_max_c,suhu_min_c,hujan_mm
0,2025-01-01,28.1,25.1,26.1
1,2025-01-02,26.7,25.0,36.7
2,2025-01-03,27.7,24.2,16.8
3,2025-01-04,27.2,24.1,21.1
4,2025-01-05,27.9,23.8,38.0


## Soal 4. Scraping etis halaman statis
Halaman: Wikipedia, *List of districts in Bali* (tabel kecamatan per kabupaten). Lisensi konten Wikipedia: CC BY-SA 4.0.

In [23]:
URL_SCRAPE = "https://en.wikipedia.org/wiki/List_of_districts_in_Bali"
LISENSI_SCRAPE = "CC BY-SA 4.0 (Wikipedia)"

# 1) Periksa robots.txt sebelum mengambil halaman
p = urlparse(URL_SCRAPE)
robots_url = f"{p.scheme}://{p.netloc}/robots.txt"
r_robots = get_aman(robots_url)
path_robots = "data/raw/robots_wikipedia.txt"
with open(path_robots, "wb") as f:
    f.write(r_robots.content)
catat_provenance("robots.txt", path_robots, robots_url, "-", "Bukti pemeriksaan kepatuhan")

rp = robotparser.RobotFileParser()
rp.parse(r_robots.text.splitlines())
boleh = rp.can_fetch(USER_AGENT, URL_SCRAPE)
print("Boleh diambil menurut robots.txt?", boleh)
assert boleh, "Dilarang oleh robots.txt, ganti halaman"

Status: 200 | https://en.wikipedia.org/robots.txt
Provenance tercatat: data/raw/robots_wikipedia.txt | MD5: fc15d80fd6a4afd9f22975e740c1b48c
Boleh diambil menurut robots.txt? True


In [24]:
# 2) Ambil halaman statis, simpan HTML mentah, lalu baca tabel
r_html = get_aman(URL_SCRAPE)
path_html = "data/raw/wikipedia_districts_bali.html"
with open(path_html, "wb") as f:
    f.write(r_html.content)
catat_provenance("Scraping halaman statis", path_html, URL_SCRAPE, LISENSI_SCRAPE, "Dicek robots.txt")

tabel_list = pd.read_html(io.StringIO(r_html.text))
for i, t in enumerate(tabel_list):
    print(i, t.shape)

Status: 200 | https://en.wikipedia.org/wiki/List_of_districts_in_Bali
Provenance tercatat: data/raw/wikipedia_districts_bali.html | MD5: a86d79586a65c4fb14085e26833cb806
0 (13, 1)
1 (10, 9)
2 (10, 6)
3 (6, 6)
4 (13, 6)
5 (10, 6)
6 (1, 2)
7 (1, 2)
8 (1, 2)
9 (9, 7)
10 (1, 2)


In [25]:
# Pilih tabel terbesar (daftar kecamatan). Ganti INDEKS_TABEL manual jika perlu.
INDEKS_TABEL = max(range(len(tabel_list)), key=lambda i: len(tabel_list[i]))
df_scrape = tabel_list[INDEKS_TABEL].copy()
if isinstance(df_scrape.columns, pd.MultiIndex):            # ratakan header bertingkat
    df_scrape.columns = ["_".join(map(str, c)).strip() for c in df_scrape.columns]
df_scrape.columns = [str(c).strip().lower().replace(" ", "_") for c in df_scrape.columns]
df_scrape.to_csv("data/interim/tabel_kecamatan_bali.csv", index=False)
print("Tabel dipilih:", INDEKS_TABEL, df_scrape.shape)
df_scrape.head()

Tabel dipilih: 0 (13, 1)


,0
0,This article is part of a series on
1,Subdivisions of Indonesia
2,Level 1
3,Provinces (provinsi or daerah istimewa or daer...
4,Level 2


## Soal 5. Penggabungan dua sumber (berkas + API), bukti tanpa duplikasi, simpan + metadata
Berkas wisman (per negara per bulan) digabung dengan cuaca bulanan hasil API lewat kunci **bulan**.

In [26]:
# 1) Ubah data wisman ke format panjang: satu baris = negara x bulan
df_long = df_file.melt(id_vars=["kawasan", "negara"], value_vars=BULAN_KOL,
                       var_name="bulan_nama", value_name="kunjungan")
df_long["bulan"] = df_long["bulan_nama"].map({b: i + 1 for i, b in enumerate(BULAN_KOL)})

# 2) Ringkas cuaca harian menjadi cuaca bulanan (satu baris per bulan)
df_api["bulan"] = df_api["tanggal"].dt.month
cuaca_bulanan = df_api.groupby("bulan").agg(
    suhu_max_rata2_c=("suhu_max_c", "mean"),
    suhu_min_rata2_c=("suhu_min_c", "mean"),
    hujan_total_mm=("hujan_mm", "sum"),
    hari_hujan=("hujan_mm", lambda s: int((s >= 1).sum())),
).round(2).reset_index()
print(cuaca_bulanan)

# 3) Merge. validate="m:1": banyak baris wisman cocok ke tepat satu baris cuaca per bulan
gabung = df_long.merge(cuaca_bulanan, on="bulan", how="left", validate="m:1")
print(gabung.shape)

    bulan  suhu_max_rata2_c  suhu_min_rata2_c  hujan_total_mm  hari_hujan
0       1             28.71             25.20           489.5          30
1       2             29.26             25.29           299.6          24
2       3             29.94             25.91           218.8          27
3       4             30.12             25.61            89.0          14
4       5             29.50             25.63           174.2          27
5       6             28.91             25.59            97.9          19
6       7             28.13             24.80            55.5          20
7       8             28.82             24.91            35.9          14
8       9             29.28             25.00            97.4          16
9      10             30.36             25.44            57.9          17
10     11             30.32             25.73           132.1          22
11     12             30.83             26.07           141.1          22
(2220, 9)


In [27]:
# Bukti tidak ada duplikasi baris
print("Baris wisman (long):", len(df_long))
print("Baris hasil merge  :", len(gabung))
print("Baris duplikat penuh:", gabung.duplicated().sum())
print("Duplikat kunci (negara,bulan):", gabung.duplicated(["negara", "bulan"]).sum())
print("Nilai cuaca kosong :", int(gabung["hujan_total_mm"].isna().sum()))
assert len(gabung) == len(df_long), "Jumlah baris berubah: ada duplikasi akibat merge"
assert gabung.duplicated().sum() == 0
assert gabung.duplicated(["negara", "bulan"]).sum() == 0
print("OK: tidak ada duplikasi baris")

path_out = "data/processed/wisman_bali_2025_cuaca.csv"
gabung.to_csv(path_out, index=False)
gabung.head()

Baris wisman (long): 2220
Baris hasil merge  : 2220
Baris duplikat penuh: 0
Duplikat kunci (negara,bulan): 0
Nilai cuaca kosong : 0
OK: tidak ada duplikasi baris


,kawasan,negara,bulan_nama,kunjungan,bulan,suhu_max_rata2_c,suhu_min_rata2_c,hujan_total_mm,hari_hujan
0,ASIA PACIFIC EXCL. ASEAN,Australian,jan,134483,1,28.71,25.2,489.5,30
1,ASIA PACIFIC EXCL. ASEAN,Bhutan,jan,0,1,28.71,25.2,489.5,30
2,ASIA PACIFIC EXCL. ASEAN,Bangladesh,jan,889,1,28.71,25.2,489.5,30
3,ASIA PACIFIC EXCL. ASEAN,Fiji,jan,0,1,28.71,25.2,489.5,30
4,ASIA PACIFIC EXCL. ASEAN,Hongkong,jan,3181,1,28.71,25.2,489.5,30


In [28]:
# Metadata hasil penggabungan
metadata = {
    "judul": "Kedatangan wisman ke Bali per negara per bulan 2025 + cuaca bulanan",
    "dibuat_oleh": NAMA, "nim": NIM,
    "tanggal_dibuat": dt.datetime.now().isoformat(timespec="seconds"),
    "sumber": ["Dinas Pariwisata Prov. Bali, Statistik Pariwisata 2025 (berkas, sheet tab7ok)",
               "Open-Meteo archive API (cuaca harian Bali 2025, diagregasi bulanan)",
               "Wikipedia List of districts in Bali (scraping, Soal 4; tidak digabung)"],
    "jumlah_baris": int(gabung.shape[0]), "jumlah_kolom": int(gabung.shape[1]),
    "kolom": list(map(str, gabung.columns)),
    "metode_gabung": "left merge pada kunci bulan, validate=m:1",
    "md5_hasil": md5_file(path_out),
    "catatan": "Data mentah tidak diubah; data wisman bersifat sementara; lihat docs/provenance.csv",
}
with open("docs/metadata_gabungan.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(json.dumps(metadata, indent=2, ensure_ascii=False))

{
  "judul": "Kedatangan wisman ke Bali per negara per bulan 2025 + cuaca bulanan",
  "dibuat_oleh": "Ach. Rafli alfian syah",
  "nim": "2501010048",
  "tanggal_dibuat": "2026-11-19T02:25:43",
  "sumber": [
    "Dinas Pariwisata Prov. Bali, Statistik Pariwisata 2025 (berkas, sheet tab7ok)",
    "Open-Meteo archive API (cuaca harian Bali 2025, diagregasi bulanan)",
    "Wikipedia List of districts in Bali (scraping, Soal 4; tidak digabung)"
  ],
  "jumlah_baris": 2220,
  "jumlah_kolom": 9,
  "kolom": [
    "kawasan",
    "negara",
    "bulan_nama",
    "kunjungan",
    "bulan",
    "suhu_max_rata2_c",
    "suhu_min_rata2_c",
    "hujan_total_mm",
    "hari_hujan"
  ],
  "metode_gabung": "left merge pada kunci bulan, validate=m:1",
  "md5_hasil": "272b323a94dade137eab07b03bc685d2",
  "catatan": "Data mentah tidak diubah; data wisman bersifat sementara; lihat docs/provenance.csv"
}


In [29]:
# Rekap akhir: isi provenance dan struktur folder
display(pd.read_csv(PROV_PATH))
for root, _, files in os.walk("."):
    if root.startswith(("./data", "./docs")):
        for fn in files:
            print(os.path.join(root, fn))

,sumber,file,url,tanggal_akses,lisensi,md5,keterangan
0,Dataset berkas (Statistik Pariwisata Bali 2025),data/raw/Table-1-7-Final-5.xls,https://disparda.baliprov.go.id/category/data-...,2026-11-19 02:25:02,Tidak ada lisensi eksplisit di berkas/portal; ...,7ff6f254fe4d4c2916f6b1b5cd3c2c43,Sheet tab7ok; unduhan manual; ada catatan *TEM...
1,API Open-Meteo (archive),data/raw/openmeteo_bali_2025.json,https://archive-api.open-meteo.com/v1/archive?...,2026-11-19 02:25:24,"CC BY 4.0 (Open-Meteo, penggunaan non-komersia...",0597cf5846cdefbe0039f1b542b184ba,Cuaca harian 2025
2,robots.txt,data/raw/robots_wikipedia.txt,https://en.wikipedia.org/robots.txt,2026-11-19 02:25:31,-,fc15d80fd6a4afd9f22975e740c1b48c,Bukti pemeriksaan kepatuhan
3,Scraping halaman statis,data/raw/wikipedia_districts_bali.html,https://en.wikipedia.org/wiki/List_of_district...,2026-11-19 02:25:34,CC BY-SA 4.0 (Wikipedia),a86d79586a65c4fb14085e26833cb806,Dicek robots.txt


./docs/provenance.csv
./docs/identifikasi_dataset.csv
./docs/data_ethics_checklist.csv
./docs/metadata_gabungan.json
./data/processed/wisman_bali_2025_cuaca.csv
./data/raw/robots_wikipedia.txt
./data/raw/Table-1-7-Final-5.xls
./data/raw/wikipedia_districts_bali.html
./data/raw/openmeteo_bali_2025.json
./data/interim/tabel_kecamatan_bali.csv
./data/interim/wisman_bali_2025_wide.csv
./data/interim/cuaca_bali_2025_harian.csv
